Purpose of this notebook:
- testing out the Wikidata query service with the python sparqlwrapper library 
- using/adjusting the code from this [gitlab jupyter notebook](https://gitlab.wikimedia.org/-/snippets/257)

In [1]:
## import libraries

from SPARQLWrapper import SPARQLWrapper, JSON 
import polars as pl
import requests
import pandas as pd

In [2]:
# Set configs

url = "https://query.wikidata.org/sparql"
user_agent = 'kg-digital-humanities/0.1.0 (https://github.com/FabiLochner/kg-digital-humanities; fabian.lochner@uni-oldenburg.de)'

### 1) Use requests library to debug 429 error codes

In [ ]:
# Query from the wikidata tutorial in notebooks/tutorial_wikidata_sparql.ipynb
query = """
# Brazilians who are poets 
SELECT ?item ?itemLabel WHERE {
  ?item wdt:P31 wd:Q5. # humans
  ?item wdt:P27 wd:Q155. # citizenship - Brazil
  ?item  wdt:P106 wd:Q49757. #occupation - poet
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" }
}
LIMIT 10
"""

In [9]:
resp = requests.get(
    url, 
    params= {"query": query, "format": "json"},
    headers = {"User-Agent": user_agent, "Accept": "application/sparql-results+json"}

)

print(f"Status: {resp.status_code}")
print(f"Retry-After: {resp.headers.get('Retry-After')}")
print(f"Body preview: {resp.text[:500]}")

Status: 200
Retry-After: None
Body preview: {
  "head" : {
    "vars" : [ "item", "itemLabel" ]
  },
  "results" : {
    "bindings" : [ {
      "item" : {
        "type" : "uri",
        "value" : "http://www.wikidata.org/entity/Q13012"
      },
      "itemLabel" : {
        "xml:lang" : "en",
        "type" : "literal",
        "value" : "Guimarães Rosa"
      }
    }, {
      "item" : {
        "type" : "uri",
        "value" : "http://www.wikidata.org/entity/Q23945"
      },
      "itemLabel" : {
        "xml:lang" : "en",
        "typ


In [ ]:
# Convert json output into pandas df

data = resp.json()

# Keys are defined by W3C SPARQL 1.1 Query Results JSON Format
# columns: item | itemLabel
rows = [
    {var: binding[var]["value"] for var in binding}
    for binding in data["results"]["bindings"]
]

df = pd.DataFrame(rows)
print(df.shape)
df.head()

(10, 2)


,item,itemLabel
0,http://www.wikidata.org/entity/Q13012,Guimarães Rosa
1,http://www.wikidata.org/entity/Q23945,Ana Cristina Cesar
2,http://www.wikidata.org/entity/Q184440,Jorge Amado
3,http://www.wikidata.org/entity/Q189042,Rubem Alves
4,http://www.wikidata.org/entity/Q199644,Adolfo Caminha


In [ ]:
# columns: item.type | item.value | itemLabel.type | itemLabel.value | itemLabel.xml:lang
df = pd.json_normalize(data["results"]["bindings"])
df.head()

,item.type,item.value,itemLabel.xml:lang,itemLabel.type,itemLabel.value
0,uri,http://www.wikidata.org/entity/Q13012,en,literal,Guimarães Rosa
1,uri,http://www.wikidata.org/entity/Q23945,en,literal,Ana Cristina Cesar
2,uri,http://www.wikidata.org/entity/Q184440,en,literal,Jorge Amado
3,uri,http://www.wikidata.org/entity/Q189042,en,literal,Rubem Alves
4,uri,http://www.wikidata.org/entity/Q199644,en,literal,Adolfo Caminha


### 2) Trying out the example of the newbook

In [12]:
sparql = SPARQLWrapper(url, agent = user_agent)

# Query from the wikidata tutorial in notebooks/tutorial_wikidata_sparql.ipynb
sparql.setQuery(query = """
# Brazilians who are poets 
SELECT ?item ?itemLabel WHERE {
  ?item wdt:P31 wd:Q5. # humans
  ?item wdt:P27 wd:Q155. # citizenship - Brazil
  ?item  wdt:P106 wd:Q49757. #occupation - poet
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" }
}
LIMIT 10
""")
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

HTTPError: HTTP Error 429: Aggressively rate-limiting to 1 req / min - this rule was created during active wdqs outage (797a132)

- [Github Wikidata two phase extractor](https://gist.github.com/ArloDune/117ee69e72c1ceaae1c45818e5d646fa)
- [Official wikidata rate limits](https://www.mediawiki.org/wiki/Wikidata_Query_Service/User_Manual#Query_limits)

In [18]:
# Trying to debug the causes
## query is exactly the same
## user_agent is also the same:

sparql = SPARQLWrapper(url, agent=user_agent)
print(sparql.agent)  

kg-digital-humanities/0.1.0 (https://github.com/FabiLochner/kg-digital-humanities; fabian.lochner@uni-oldenburg.de)


### 3) Trying out own sparql project query with requests

In [3]:
# Query from the wikidata tutorial in notebooks/tutorial_wikidata_sparql.ipynb
query = """
# B20 German writers OR novelists OR poets with some exammple properties
SELECT ?item ?itemLabel ?sexLabel ?dobLabel ?dodLabel 
WHERE {
  ?item wdt:P31 wd:Q5. # humans
  
  # German citizenship OR writes in German language 
  {?item wdt:P27 wd:Q183.} # citizenship - Germany
  UNION
  {?item wdt:P6886 wd:Q188.} # language written - German
  
  # occupation: writer OR novelist OR poet                                  
  {?item  wdt:P106 wd:Q36180.} #occupation - writers
  UNION
  {?item  wdt:P106 wd:Q6625963 .} #occupation - novelist
  UNION
  {?item wdt:P106 wd:Q49757 .} # occupation - poet

  OPTIONAL {?item wdt:P21 ?sex }. #put the sex/gender into a variable
  OPTIONAL {?item wdt:P569 ?dob} .#put the date of birth into a variable
  OPTIONAL {?item wdt:P570 ?dod }. #put the date of death into a variable

  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" }
}
LIMIT 50
"""

In [4]:
resp = requests.get(
    url, 
    params= {"query": query, "format": "json"},
    headers = {"User-Agent": user_agent, "Accept": "application/sparql-results+json"}

)

print(f"Status: {resp.status_code}")
print(f"Retry-After: {resp.headers.get('Retry-After')}")
print(f"Body preview: {resp.text[:500]}")

Status: 200
Retry-After: None
Body preview: {
  "head" : {
    "vars" : [ "item", "itemLabel", "sexLabel", "dobLabel", "dodLabel" ]
  },
  "results" : {
    "bindings" : [ {
      "item" : {
        "type" : "uri",
        "value" : "http://www.wikidata.org/entity/Q38873"
      },
      "itemLabel" : {
        "xml:lang" : "en",
        "type" : "literal",
        "value" : "Lou Andreas-Salomé"
      },
      "sexLabel" : {
        "xml:lang" : "en",
        "type" : "literal",
        "value" : "female"
      },
      "dobLabel" : {
    


In [6]:
# Convert json output into pandas df

data = resp.json()

# Keys are defined by W3C SPARQL 1.1 Query Results JSON Format
# columns: item | itemLabel
rows = [
    {var: binding[var]["value"] for var in binding}
    for binding in data["results"]["bindings"]
]

df = pd.DataFrame(rows)
print(df.shape)
df.head(20)

(50, 5)


,item,itemLabel,sexLabel,dobLabel,dodLabel
0,http://www.wikidata.org/entity/Q38873,Lou Andreas-Salomé,female,1861-02-12T00:00:00Z,1937-02-05T00:00:00Z
1,http://www.wikidata.org/entity/Q41309,Franz Liszt,male,1811-10-22T00:00:00Z,1886-07-31T00:00:00Z
2,http://www.wikidata.org/entity/Q42747,Heinrich Böll,male,1917-12-21T00:00:00Z,1985-07-16T00:00:00Z
3,http://www.wikidata.org/entity/Q42831,Ivan Turgenev,male,1818-11-09T00:00:00Z,1883-09-03T00:00:00Z
4,http://www.wikidata.org/entity/Q43166,Paul Zech,male,1881-02-19T00:00:00Z,1946-09-07T00:00:00Z
5,http://www.wikidata.org/entity/Q43523,Gerhart Hauptmann,male,1862-11-15T00:00:00Z,1946-06-06T00:00:00Z
6,http://www.wikidata.org/entity/Q43833,Gregor Riegler,male,1950-11-23T00:00:00Z,NaN
7,http://www.wikidata.org/entity/Q43833,Gregor Riegler,male,1959-11-23T00:00:00Z,NaN
8,http://www.wikidata.org/entity/Q44003,Theodor Herzl,male,1860-05-02T00:00:00Z,1904-07-03T00:00:00Z
9,http://www.wikidata.org/entity/Q44107,Peter Handke,male,1942-12-06T00:00:00Z,NaN
